In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 4 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251213_110508.csv
Loaded: NBA_DFS_20251213_110106.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Jalen Brunson,Over,29.5,-137,2025-12-13,2025-12-13T19:00:00Z,2025-12-13 11:01:06
1,PrizePicks,player_points,Jalen Brunson,Under,29.5,-137,2025-12-13,2025-12-13T19:00:00Z,2025-12-13 11:01:06
2,PrizePicks,player_points,Paolo Banchero,Over,22.5,-137,2025-12-13,2025-12-13T19:00:00Z,2025-12-13 11:01:06
3,PrizePicks,player_points,Paolo Banchero,Under,22.5,-137,2025-12-13,2025-12-13T19:00:00Z,2025-12-13 11:01:06
4,PrizePicks,player_points,Karl-Anthony Towns,Over,21.5,-137,2025-12-13,2025-12-13T19:00:00Z,2025-12-13 11:01:06


In [4]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "fg3a_model": "src/models/saved/fg3a_model.pkl",
    "fta_model": "src/models/saved/fta_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## For Post Analysis

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

singleBets = calculateSingleBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,                  
    max_player_appearances=1,  
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)
singleBets.to_csv(f'notebooks/exploration/old_evs/singleBets_UD_{current_date}.csv', index=False)
singleBets

Computing predictions for 16 players...
Found 16 valid players


,NAME,LINE,SIDE,PREDICTION,MODEL_PROB,IMPLIED_PROB,EDGE,BET_EDGE,ODDS,DECIMAL_ODDS,EV,EV_PERCENT,KELLY_QUARTER,TEAM,OPPONENT
14,Keldon Johnson,8.5,over,14.71,0.953,0.49,6.21,0.463,-102,1.980,0.8874,88.74,0.2263,SAS,OKC
9,Shai Gilgeous-Alexander,31.5,under,17.07,0.939,0.50,14.43,0.439,-103,1.971,0.8508,85.08,0.2191,OKC,SAS
13,Alex Caruso,4.5,over,8.62,0.932,0.53,4.12,0.402,-120,1.833,0.7096,70.96,0.2129,OKC,SAS
1,Jalen Brunson,29.5,under,17.65,0.892,0.51,11.85,0.382,-110,1.909,0.7035,70.35,0.1935,NYK,ORL
6,Anthony Black,14.5,under,9.73,0.792,0.47,4.77,0.322,108,2.080,0.6482,64.82,0.1500,ORL,NYK
5,Mitchell Robinson,3.5,over,5.36,0.800,0.50,1.86,0.300,-105,1.952,0.5627,56.27,0.1477,NYK,ORL
15,Luke Kornet,4.5,over,5.79,0.724,0.50,1.29,0.224,-105,1.952,0.4131,41.31,0.1084,SAS,OKC
3,Jordan Clarkson,10.5,under,6.75,0.685,0.49,3.75,0.195,100,2.000,0.3709,37.09,0.0927,NYK,ORL
2,OG Anunoby,14.5,over,16.72,0.696,0.53,2.22,0.166,-120,1.833,0.2752,27.52,0.0826,NYK,ORL
11,Jalen Williams,19.5,under,16.43,0.612,0.52,3.07,0.092,-115,1.870,0.1438,14.38,0.0413,OKC,SAS


In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=100,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)


prizepicksPairs.to_csv(f'notebooks/exploration/old_evs/prizepicksPairs_{current_date}.csv', index=False)
# prizepicksPairs.to_csv(f'notebooks/exploration/old_evs/underdogPairs_{current_date}_{today}.csv', index=False)
prizepicksPairs

Computing predictions for 111 players...


KeyboardInterrupt: 

## Top EVs for 2 leg bets

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs

Computing predictions for 76 players...
Found 73 valid players
Generated 2355 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
2307,Desmond Bane,Stephon Castle,20.5,14.5,-110,-121,7.22,21.88,0.976,0.959,under,over,0.936,249,226.53,0.2274
170,Jalen Johnson,Dwight Powell,24.5,3.5,-118,-105,12.23,6.77,0.935,0.894,under,over,0.836,261,201.84,0.1933
1653,Ja Morant,Ajay Mitchell,19.5,9.5,-108,-111,24.39,14.25,0.879,0.908,over,over,0.798,266,192.09,0.1805
1741,Kentavious Caldwell-Pope,Shai Gilgeous-Alexander,7.5,31.5,-105,-121,10.79,21.12,0.831,0.934,over,under,0.777,257,177.22,0.1724
372,Onyeka Okongwu,Rudy Gobert,17.5,11.5,-110,-105,10.10,14.82,0.840,0.820,under,over,0.688,273,156.81,0.1436
399,Mouhamed Gueye,Isaac Okoro,3.5,6.5,-115,-102,6.40,9.03,0.832,0.785,over,over,0.654,270,141.93,0.1314
466,Miles Bridges,Justin Champagnie,22.5,9.5,-112,100,16.75,11.68,0.805,0.772,under,over,0.622,279,135.56,0.1215
2236,Naz Reid,Jordan Clarkson,15.5,10.5,110,-104,10.84,13.33,0.715,0.775,under,over,0.555,312,128.59,0.1030
2295,Karl-Anthony Towns,Harrison Barnes,21.5,10.5,-112,-106,15.12,5.18,0.793,0.773,under,under,0.613,268,125.66,0.1172
1122,Kyshawn George,Jalen Suggs,15.5,17.5,-115,-114,18.97,11.28,0.802,0.800,over,under,0.641,251,125.04,0.1245


### Prizepicks picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

prizepicksPairs = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
prizepicksPairs.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Computing predictions for 130 players...
Found 126 valid players
Generated 7046 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
4597,Jared McCain,Desmond Bane,13.5,20.5,-105,-110,1.84,7.22,0.976,0.976,under,under,0.953,273,255.30,0.2338
220,Jalen Johnson,Ajay Mitchell,24.5,9.5,-118,-111,12.23,14.25,0.935,0.908,under,over,0.849,251,198.02,0.1972
6777,De'Anthony Melton,Shai Gilgeous-Alexander,8.0,31.5,-137,-121,15.95,21.12,0.981,0.934,over,under,0.916,216,189.52,0.2194
5092,Ja Morant,Michael Porter Jr.,19.5,23.5,-108,105,24.39,15.54,0.879,0.816,over,under,0.717,295,183.28,0.1553
4179,Joel Embiid,Stephon Castle,23.5,15.0,-110,-137,16.63,21.88,0.866,0.946,under,over,0.819,230,170.40,0.1852
2965,Isaac Okoro,Kentavious Caldwell-Pope,5.5,7.5,-115,-105,9.03,10.79,0.871,0.831,over,over,0.724,265,164.16,0.1549
2892,Liam McNeeley,Rudy Gobert,6.5,11.5,-125,-105,1.37,14.82,0.889,0.820,under,over,0.729,251,155.93,0.1553
1779,Brandon Miller,Andrew Nembhard,22.0,15.5,-137,-113,13.66,19.25,0.915,0.828,under,over,0.758,226,146.96,0.1626
1259,Caris LeVert,Pascal Siakam,8.5,23.5,100,-105,3.78,17.80,0.773,0.794,under,under,0.614,290,139.29,0.1201
1675,Miles Bridges,Justin Champagnie,22.5,9.5,-112,100,16.75,11.68,0.805,0.772,under,over,0.622,279,135.56,0.1215


## 3 leg parlay

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Computing predictions for 76 players...
Found 73 valid players
Generated 44075 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
43154,Dwight Powell,Desmond Bane,Stephon Castle,3.5,20.5,14.5,-105,-110,-121,6.77,7.22,21.88,0.894,0.976,0.959,over,under,over,0.837,581,469.86,0.2022
4977,Jalen Johnson,Ja Morant,Ajay Mitchell,24.5,19.5,9.5,-118,-108,-111,12.23,24.39,14.25,0.935,0.879,0.908,under,over,over,0.746,576,404.33,0.1755
10744,Onyeka Okongwu,Kentavious Caldwell-Pope,Shai Gilgeous-Alexander,17.5,7.5,31.5,-110,-105,-121,10.10,10.79,21.12,0.840,0.831,0.934,under,over,under,0.652,581,344.13,0.1481
11574,Mouhamed Gueye,Isaac Okoro,Rudy Gobert,3.5,6.5,11.5,-115,-102,-105,6.40,9.03,14.82,0.832,0.785,0.820,over,over,over,0.536,623,287.54,0.1154
13498,Miles Bridges,Justin Champagnie,Jordan Clarkson,22.5,9.5,10.5,-112,100,-104,16.75,11.68,13.33,0.805,0.772,0.775,under,over,over,0.482,643,258.11,0.1004


### Prizepicks picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Computing predictions for 130 players...
Found 126 valid players
Generated 229516 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
8507,Jalen Johnson,Jared McCain,Desmond Bane,24.5,13.5,20.5,-118,-105,-110,12.23,1.84,7.22,0.935,0.976,0.976,under,under,under,0.890,589,513.53,0.2180
203197,Ja Morant,De'Anthony Melton,Ajay Mitchell,19.5,8.0,9.5,-108,-137,-111,24.39,15.95,14.25,0.879,0.981,0.908,over,over,over,0.783,533,395.48,0.1855
176103,Joel Embiid,Michael Porter Jr.,Shai Gilgeous-Alexander,23.5,23.5,31.5,-110,105,-121,16.63,15.54,21.12,0.866,0.816,0.934,under,under,under,0.661,615,372.33,0.1514
135802,Isaac Okoro,Kentavious Caldwell-Pope,Stephon Castle,5.5,7.5,15.0,-115,-105,-137,9.03,10.79,21.88,0.871,0.831,0.946,over,over,over,0.684,531,331.87,0.1562
130409,Liam McNeeley,Andrew Nembhard,Rudy Gobert,6.5,15.5,11.5,-125,-113,-105,1.37,19.25,14.82,0.889,0.828,0.820,under,over,over,0.604,562,299.52,0.1332
